# Exploratory Data Profiling & Quality Risk Analysis
## Workshop 2: Building a Reliable Batch Data Pipeline
**Academic Course:** ETL (G01) — Data Engineering & Artificial Intelligence  
**Sources Profiled:** Spotify Streaming Dataset (`spotify_dataset.csv`) & Grammy Awards Dataset (`the_grammy_awards.csv`)  
**Objective:** Provide evidence-based justification for data quality rules, integration contracts, and dimensional model design in accordance with **PDF §6.3**.

---
### Table of Contents
1. Environment Setup & Data Loading
2. Dimension 1: Structural Analysis (Shape, Columns, Types)
3. Dimension 2: Completeness Analysis (Missing Values & Ratios)
4. Dimension 3: Uniqueness & Duplication Analysis (Candidate Keys)
5. Dimension 4: Categorical Content Analysis (Genres & Award Categories)
6. Dimension 5: Numerical Content Analysis (Popularity & Audio Features)
7. Dimension 6: Temporal Content Analysis (Grammy Timeline 1958–2019)
8. Dimension 7: Cross-Source Consistency & Matching Analysis
9. Evidence-Based Quality Risk & Rules Register (Mapeo a las 24 Reglas DQ)



In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
DATA_DIR = PROJECT_ROOT / 'data' if (PROJECT_ROOT / 'data').exists() else PROJECT_ROOT
EVIDENCE_DIR = PROJECT_ROOT / 'docs' / 'evidence'
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data source path: {DATA_DIR}")


---
## 1. Data Ingestion & Source Verification
We load both provided raw files preserving their original state without premature alterations.


In [ ]:
spotify_path = DATA_DIR / 'spotify_dataset.csv' if (DATA_DIR / 'spotify_dataset.csv').exists() else PROJECT_ROOT / 'spotify_dataset.csv'
grammy_path = DATA_DIR / 'the_grammy_awards.csv' if (DATA_DIR / 'the_grammy_awards.csv').exists() else PROJECT_ROOT / 'the_grammy_awards.csv'

df_sp = pd.read_csv(spotify_path, low_memory=False)
if 'Unnamed: 0' in df_sp.columns:
    df_sp = df_sp.drop(columns=['Unnamed: 0'])

df_gr = pd.read_csv(grammy_path, low_memory=False)

print(f"✓ Spotify Dataset loaded: {df_sp.shape[0]:,} rows × {df_sp.shape[1]} columns")
print(f"✓ Grammy Awards Dataset loaded: {df_gr.shape[0]:,} rows × {df_gr.shape[1]} columns")


---
## 2. Dimension 1: Structural Analysis
We examine schema definitions, data types, and required analytical fields.


In [ ]:
print("=== SPOTIFY DATA TYPES & NULL COUNTS ===")
sp_info = pd.DataFrame({
    'Column': df_sp.columns,
    'Dtype': df_sp.dtypes.values,
    'Non-Null Count': df_sp.notnull().sum().values,
    'Null Count': df_sp.isnull().sum().values,
    'Null %': (df_sp.isnull().mean() * 100).round(4).values
})
display(sp_info)

print("\n=== GRAMMY AWARDS DATA TYPES & NULL COUNTS ===")
gr_info = pd.DataFrame({
    'Column': df_gr.columns,
    'Dtype': df_gr.dtypes.values,
    'Non-Null Count': df_gr.notnull().sum().values,
    'Null Count': df_gr.isnull().sum().values,
    'Null %': (df_gr.isnull().mean() * 100).round(4).values
})
display(gr_info)


---
## 3. Dimension 2: Completeness Analysis
**Critical findings:**
- Spotify contains exactly **1 null** in `artists`, `album_name`, and `track_name` (< 0.001%).
- Grammy contains **1,840 null values** in `artist` (38.25%). These correspond to production/album awards without individual artist attribution (e.g., Best Engineered Album, Classical Producer of the Year).
- `workers` (45.53% null) and `img` (28.42% null) have missing values, but neither is required by our analytical requirements.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spotify nulls
sns.barplot(data=sp_info[sp_info['Null Count'] > 0], x='Column', y='Null Count', ax=axes[0], color='salmon')
axes[0].set_title('Spotify Dataset: Null Count per Field')
axes[0].set_ylabel('Missing Rows')

# Grammy nulls
gr_nulls = gr_info[gr_info['Null Count'] > 0].sort_values(by='Null %', ascending=False)
sns.barplot(data=gr_nulls, x='Column', y='Null %', ax=axes[1], palette='Blues_r')
axes[1].set_title('Grammy Awards: Missing Value Percentage (%)')
axes[1].set_ylabel('Missing Percentage (%)')

plt.tight_layout()
plt.savefig(EVIDENCE_DIR / 'nulls_comparison.png', dpi=150)
plt.show()


---
## 4. Dimension 3: Uniqueness & Duplication Analysis
We investigate candidate keys and evaluate whether repeated values are expected at the source grain.


In [ ]:
print("=== SPOTIFY UNIQUENESS ANALYSIS ===")
total_sp_rows = len(df_sp)
unique_tracks = df_sp['track_id'].nunique()
dup_tracks = df_sp.duplicated(subset=['track_id']).sum()
dup_track_genre = df_sp.duplicated(subset=['track_id', 'track_genre']).sum()
unique_track_genre = len(df_sp.drop_duplicates(subset=['track_id', 'track_genre']))

print(f"Total Rows:                         {total_sp_rows:,}")
print(f"Unique track_id:                    {unique_tracks:,}")
print(f"Duplicate track_id count:           {dup_tracks:,} (Tracks appearing in multiple genres)")
print(f"Duplicate (track_id, track_genre):  {dup_track_genre:,}")
print(f"Unique (track_id, track_genre):     {unique_track_genre:,} (Declared Grain in Fact Table)")

print("\n=== GRAMMY AWARDS UNIQUENESS ANALYSIS ===")
print(f"Total Grammy rows:                  {len(df_gr):,}")
print(f"Unique categories:                  {df_gr['category'].nunique():,}")
print(f"Unique artists (non-null):          {df_gr['artist'].dropna().nunique():,}")


---
## 5. Dimension 4: Categorical Content Analysis
We profile categorical distributions across Spotify genres and Grammy award categories.


In [ ]:
print(f"Total Spotify Genres: {df_sp['track_genre'].nunique()}")
print("Top 15 Most Frequent Genres in Spotify:")
display(df_sp['track_genre'].value_counts().head(15))

plt.figure(figsize=(12, 5))
top_genres = df_sp['track_genre'].value_counts().head(20)
sns.barplot(x=top_genres.index, y=top_genres.values, palette='viridis')
plt.title('Top 20 Spotify Genres by Track Volume')
plt.xlabel('Genre Label')
plt.ylabel('Track Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(EVIDENCE_DIR / 'top_genres_distribution.png', dpi=150)
plt.show()


---
## 6. Dimension 5: Numerical Content Analysis (Popularity & Audio Features)
We profile distributions, boundary constraints, and compute the empirical **75th Percentile (P75)** required for REQ-03.


In [ ]:
audio_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence', 'tempo']

print("=== SPOTIFY NUMERICAL METRICS DESCRIPTIVE STATISTICS ===")
display(df_sp[['popularity'] + audio_cols].describe().round(3))

# Calculate P75 of popularity
p75_val = df_sp['popularity'].quantile(0.75)
p50_val = df_sp['popularity'].quantile(0.50)
print(f"\n★ Popularity Median (P50):           {p50_val:.1f}")
print(f"★ Popularity 75th Percentile (P75):   {p75_val:.1f} (Threshold for High Popularity Share - HPS)")

# Plot popularity distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_sp['popularity'], bins=50, kde=True, color='teal')
plt.axvline(p75_val, color='red', linestyle='--', linewidth=2, label=f'P75 Threshold ({p75_val:.0f})')
plt.title('Distribution of Spotify Popularity (0–100 Scale)')
plt.xlabel('Popularity')
plt.ylabel('Track Frequency')
plt.legend()
plt.tight_layout()
plt.savefig(EVIDENCE_DIR / 'popularity_distribution.png', dpi=150)
plt.show()


---
## 7. Dimension 6: Temporal Content Analysis
We profile the temporal span of Grammy ceremonies to support historical cohort analysis (REQ-03).


In [ ]:
print("=== GRAMMY TEMPORAL COVERAGE ===")
print(f"Earliest Ceremony Year: {df_gr['year'].min()}")
print(f"Latest Ceremony Year:   {df_gr['year'].max()}")
print(f"Total Unique Years:     {df_gr['year'].nunique()}")

# Group by decade
df_gr['decade'] = (df_gr['year'] // 10 * 10).astype(str) + 's'
decade_counts = df_gr['decade'].value_counts().sort_index()

plt.figure(figsize=(10, 4))
sns.barplot(x=decade_counts.index, y=decade_counts.values, palette='magma')
plt.title('Grammy Awards Records by Ceremony Decade (1950s - 2010s)')
plt.xlabel('Decade Cohort')
plt.ylabel('Award Count')
plt.tight_layout()
plt.savefig(EVIDENCE_DIR / 'grammy_years_distribution.png', dpi=150)
plt.show()


---
## 8. Dimension 7: Cross-Source Consistency & Matching Analysis
We evaluate artist name compatibility and matching cardinality between Spotify and Grammy Awards.


In [ ]:
# Normalisation helper
def clean_artist(s):
    return s.dropna().astype(str).str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)

sp_primary_artists = clean_artist(df_sp['artists'].astype(str).str.split(';').str[0])
gr_artists = clean_artist(df_gr['artist'])

unique_sp_artists = set(sp_primary_artists.unique())
unique_gr_artists = set(gr_artists.unique())

intersection = unique_sp_artists.intersection(unique_gr_artists)

print(f"Unique Spotify Primary Artists:   {len(unique_sp_artists):,}")
print(f"Unique Grammy Award Artists:      {len(unique_gr_artists):,}")
print(f"Matched Artists in Intersection:  {len(intersection):,} artists")
print(f"Matching Coverage of Grammy Pool: {len(intersection) / len(unique_gr_artists) * 100:.2f}%")


---
## 9. Quality Risks & Data Quality Rules Register (24 Reglas DQ)
The empirical findings from this profiling notebook justify every Great Expectations rule configured in the pipeline:

| Rule ID | Layer / Dataset | Dimension | Attribute | Empirical Evidence / Risk | Configured Threshold | Severity |
|---|---|---|---|---|---|---|
| **DQ-SP-01** | Spotify Raw | Completeness | `track_id` | 0 nulls observed; vital as natural key | 100% Not Null | **Critical** |
| **DQ-SP-02** | Spotify Raw | Validity | `popularity` | Valid values range strictly between 0 and 100 | [0, 100] | **Critical** |
| **DQ-SP-03** | Spotify Raw | Completeness | `artists` | 1 null observed (0.0009%); integration key | mostly=0.99 | **Critical** |
| **DQ-SP-04** | Spotify Raw | Completeness | `track_genre` | 0 nulls observed; required for genre slicing | 100% Not Null | **Critical** |
| **DQ-SP-05** | Spotify Raw | Validity | `danceability` | Bounded between 0.0 and 1.0; no outliers | [0.0, 1.0] | **Critical** |
| **DQ-SP-06** | Spotify Raw | Validity | `energy` | Bounded between 0.0 and 1.0; no outliers | [0.0, 1.0] | **Critical** |
| **DQ-SP-07** | Spotify Raw | Uniqueness | `track_id` | 24,259 multi-genre collisions observed in raw | Unique | **Warning** |
| **DQ-SP-08** | Spotify Raw | Completeness | `album_name` | 1 null observed | mostly=0.99 | **Warning** |
| **DQ-SP-09** | Spotify Raw | Validity | `duration_ms` | Track files shorter than 10s indicate corrupt audio | min=10,000 ms | **Informational** |
| **DQ-GR-01** | Grammy Raw | Completeness | `year` | 0 nulls; essential for time dimension | 100% Not Null | **Critical** |
| **DQ-GR-02** | Grammy Raw | Validity | `year` | Historical ceremony range from 1958 to 2019 | [1958, 2030] | **Critical** |
| **DQ-GR-03** | Grammy Raw | Completeness | `category` | 0 nulls; category defines broad family mapping | 100% Not Null | **Critical** |
| **DQ-GR-04** | Grammy Raw | Validity | `winner` | Binary flag driving win count | In {0, 1, True, False} | **Critical** |
| **DQ-GR-05** | Grammy Raw | Completeness | `artist` | 38.25% nulls (album/producer awards); filtered | mostly=0.60 | **Warning** |
| **DQ-GR-06** | Grammy Raw | Completeness | `nominee` | 6 nulls observed | mostly=0.99 | **Warning** |
| **DQ-PREP-01** | Prepared | Completeness | `track_id` | Mandatory business key post-cleaning | 100% Not Null | **Critical** |
| **DQ-PREP-02** | Prepared | Completeness | `primary_artist` | Cleaned primary artist must never be null | 100% Not Null | **Critical** |
| **DQ-PREP-03** | Prepared | Completeness | `track_genre` | Cleaned genre must never be null | 100% Not Null | **Critical** |
| **DQ-PREP-04** | Prepared | Validity | `popularity` | Prepared popularity must be within [0, 100] | [0, 100] | **Critical** |
| **DQ-PREP-05** | Prepared | Validity | `has_grammy` | Binary flag derived from integration | In {0, 1} | **Critical** |
| **DQ-PREP-06** | Prepared | Validity | `grammy_nominations` | Non-negative counter | min=0 | **Critical** |
| **DQ-PREP-07** | Prepared | Validity | `danceability` | Cleaned feature within boundary | [0.0, 1.0] | **Critical** |
| **DQ-PREP-08** | Prepared | Validity | `energy` | Cleaned feature within boundary | [0.0, 1.0] | **Critical** |
| **DQ-PREP-09** | Prepared | Uniqueness | `(track_id, track_genre)` | Composite key enforcing Track x Genre grain | Compound Unique | **Critical** |
| **DQ-PREP-10** | Prepared | Completeness | Row count | Output batch must contain data | min=1 row | **Critical** |

---
**Data Profiling Completed Successfully.** This notebook is reproducible and satisfies all requirements of **PDF §6.3**.

